In [38]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier   , DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression 
from sklearn.neighbors import KNeighborsClassifier , KNeighborsRegressor
from sklearn.ensemble import AdaBoostClassifier ,AdaBoostRegressor, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score,r2_score  
from tqdm import tqdm
import lightgbm as lgb
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.compose import make_column_transformer , make_column_selector

from xgboost import XGBClassifier , XGBRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")
import os 
os.chdir('E:\Backup Files Abhinav_Satyam\Practical Machine Learning\Sanjay Sir Git Repo\Datasets\Cases\Medical_Cost_Personal')

In [6]:
insurance = pd.read_csv('insurance.csv')
insurance

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [13]:
X,y = insurance.drop('charges', axis = 1) , insurance['charges']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3, random_state=26)

In [14]:
ohe = OneHotEncoder(sparse_output=False,drop='first').set_output(transform='pandas')
ct = make_column_transformer(
    (ohe, make_column_selector(dtype_include=object)),
    ('passthrough', make_column_selector(dtype_include=['int64','float64'])),
    verbose_feature_names_out=False
).set_output(transform='pandas')


In [21]:
X_train_trans = ct.fit_transform(X_train) 
X_test_trans = ct.transform(X_test)

# sklearn

In [27]:
 

learning_rate = np.linspace(0.01,0.9,10)
max_depth = [None,2,3,4]
n_estimator  = [25,50,100]

scores =  []
for l in tqdm(learning_rate):
    for d in max_depth:
        for n in n_estimator:

            dtc = DecisionTreeRegressor(random_state=26,max_depth=d)
            gbm = AdaBoostRegressor(random_state = 26,learning_rate=l,estimator=dtc,  n_estimators=n)
            gbm.fit(X_train_trans, y_train)
            y_pred = gbm.predict(X_test_trans) 

            scores.append([l, d, n ,r2_score(y_test, y_pred) ])
df_scores = pd.DataFrame(scores , columns=['l_rate','n','depth','score'])
df_scores.sort_values(by='score' , ascending=False)

100%|██████████| 10/10 [00:17<00:00,  1.72s/it]


,l_rate,n,depth,score
9,0.010000,4.0,25,0.864379
10,0.010000,4.0,50,0.863397
11,0.010000,4.0,100,0.862489
21,0.108889,4.0,25,0.858199
6,0.010000,3.0,25,0.856378
...,...,...,...,...
59,0.405556,4.0,100,0.715452
118,0.900000,4.0,50,0.711754
71,0.504444,4.0,100,0.702273
107,0.801111,4.0,100,0.678973


In [24]:


learning_rate = np.linspace(0.01,0.9,10)
max_depth = [None,2,3,4]
n_estimator  = [25,50,100]

scores =  []
for l in tqdm(learning_rate):
    for d in max_depth:
        for n in n_estimator:
            
            gbm = GradientBoostingRegressor(random_state = 26, verbose = False,learning_rate=l, max_depth=d, n_estimators=n)
            gbm.fit(X_train_trans, y_train)
            y_pred = gbm.predict(X_test_trans) 

            scores.append([l, d, n ,r2_score(y_test, y_pred) ])
df_scores = pd.DataFrame(scores , columns=['l_rate','n','depth','score'])
df_scores.sort_values(by='score' , ascending=False)

100%|██████████| 10/10 [00:13<00:00,  1.34s/it]


,l_rate,n,depth,score
19,0.108889,3.0,50,0.869232
30,0.207778,3.0,25,0.869119
20,0.108889,3.0,100,0.868956
18,0.108889,3.0,25,0.866720
42,0.306667,3.0,25,0.866690
...,...,...,...,...
4,0.010000,2.0,50,0.522892
9,0.010000,4.0,25,0.344310
0,0.010000,NaN,25,0.339508
6,0.010000,3.0,25,0.338345


# XGBM

In [32]:
learning_rate = np.linspace(0.01,0.9,10)
max_depth = [None,2,3,4]
n_estimator  = [25,50,100]

scores =  []
for l in tqdm(learning_rate):
    for d in max_depth:
        for n in n_estimator:
            
            gbm = XGBRegressor(random_state = 26, learning_rate=l, max_depth=d, n_estimators=n)
            gbm.fit(X_train_trans, y_train)
            y_pred = gbm.predict(X_test_trans) 

            scores.append([l, d, n ,r2_score(y_test, y_pred) ])
df_scores = pd.DataFrame(scores , columns=['learning_rate','max_depth','n_estimator','score'])
df_scores.sort_values(by='score' , ascending=False)

100%|██████████| 10/10 [00:14<00:00,  1.47s/it]


,learning_rate,max_depth,n_estimator,score
19,0.108889,3.0,50,0.872067
30,0.207778,3.0,25,0.871976
39,0.306667,2.0,25,0.871520
22,0.108889,4.0,50,0.870778
33,0.207778,4.0,25,0.870762
...,...,...,...,...
4,0.010000,2.0,50,0.522753
0,0.010000,NaN,25,0.339735
9,0.010000,4.0,25,0.338969
6,0.010000,3.0,25,0.335574


# LGBM

In [36]:
learning_rate = np.linspace(0.01,0.9,10)
max_depth = [None,2,3,4]
n_estimator  = [25,50,100]

scores =  []
for l in tqdm(learning_rate):
    for d in max_depth:
        for n in n_estimator:
            
            gbm = lgb.LGBMRegressor(random_state = 26, verbose = -1,learning_rate=l, max_depth=d, n_estimators=n)
            gbm.fit(X_train_trans, y_train)
            y_pred = gbm.predict(X_test_trans) 

            scores.append([l, d, n ,r2_score(y_test, y_pred) ])
df_scores = pd.DataFrame(scores , columns=['l_rate','n','depth','score'])
df_scores.sort_values(by='score' , ascending=False)

100%|██████████| 10/10 [00:02<00:00,  4.12it/s]


,l_rate,n,depth,score
30,0.207778,3.0,25,0.870819
31,0.207778,3.0,50,0.869950
19,0.108889,3.0,50,0.869713
17,0.108889,2.0,100,0.869398
22,0.108889,4.0,50,0.869261
...,...,...,...,...
4,0.010000,2.0,50,0.522892
0,0.010000,NaN,25,0.347231
9,0.010000,4.0,25,0.344334
6,0.010000,3.0,25,0.338345


# CAT BOOST

In [41]:
learning_rate = np.linspace(0.01,0.9,10)
max_depth = [None,2,3,4]
n_estimator  = [25,50,100]

scores =  []
for l in tqdm(learning_rate):
    for d in max_depth:
        for n in n_estimator:
            
            gbm = CatBoostRegressor(random_state = 26, verbose = False,learning_rate=l, max_depth=d, n_estimators=n)
            gbm.fit(X_train_trans, y_train)
            y_pred = gbm.predict(X_test_trans) 

            scores.append([l, d, n ,r2_score(y_test, y_pred) ])
df_scores = pd.DataFrame(scores , columns=['l_rate','n','depth','score'])
df_scores.sort_values(by='score' , ascending=False)

100%|██████████| 10/10 [00:24<00:00,  2.49s/it]


,l_rate,n,depth,score
31,0.207778,3.0,50,0.874315
99,0.801111,2.0,25,0.873775
78,0.603333,3.0,25,0.873340
34,0.207778,4.0,50,0.873296
32,0.207778,3.0,100,0.872602
...,...,...,...,...
4,0.010000,2.0,50,0.460703
9,0.010000,4.0,25,0.303635
6,0.010000,3.0,25,0.302483
0,0.010000,NaN,25,0.290878
